# Module 4 · Solutions
Attempt first. Each solution states the *message* the chart carries — because a chart without a message is decoration.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
BASE = "data/"
fin = pd.read_csv(BASE + "company_financials.csv")
cli = pd.read_csv(BASE + "client_book.csv")
px  = pd.read_csv(BASE + "nifty50_prices.csv", parse_dates=["date"])
uni = pd.read_csv(BASE + "nse_stock_universe.csv", parse_dates=["date"])

## 4A

In [ ]:
# Ex1 - stores trend with a MESSAGE title
fig, ax = plt.subplots(figsize=(8,3.5))
ax.plot(fin["fiscal_year"], fin["stores_count"], marker="o", color="#2563EB")
ax.set_title("MoneyMart doubled its store network in a decade (180 -> 400+)", loc="left", fontweight="bold")
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

# Ex2 - AUM box by segment, log scale
fig, ax = plt.subplots(figsize=(8,4))
sns.boxplot(data=cli, x="segment", y="aum_inr",
            order=["Mass","Affluent","HNI","Ultra-HNI"], ax=ax)
ax.set_yscale("log")
ax.set_title("HNI shows the widest spread - 'HNI' hides very different clients", loc="left")
plt.tight_layout(); plt.show()

# Ex3 - the chart crime, side by side
fig, axes = plt.subplots(1, 2, figsize=(10,3.5))
for a, start in zip(axes, [0, 150]):
    a.bar(fin["fiscal_year"], fin["stores_count"], color="#2563EB")
    a.set_ylim(bottom=start)
    a.set_title("Honest (from 0)" if start==0 else "CRIME (from 150) - growth exaggerated", fontsize=10)
    a.tick_params(axis='x', rotation=45, labelsize=7)
plt.tight_layout(); plt.show()
print("Bars encode value by LENGTH; truncate the axis and the lengths lie. Lines encode by POSITION, so a truncated line axis can be legitimate.")

## 4B

In [ ]:
# Ex1 - COVID-year waterfall: which cost shrank least?
yr = fin[fin["fiscal_year"]=="FY20-21"].iloc[0]
prev = fin[fin["fiscal_year"]=="FY19-20"].iloc[0]
for c_ in ["cogs_cr","employee_cost_cr","marketing_cr","other_opex_cr"]:
    chg = yr[c_]/prev[c_] - 1
    print(f"{c_:<20} {chg*100:+6.1f}% (revenue fell {yr['revenue_cr']/prev['revenue_cr']-1:+.1%})")
print("Employee costs fall least -> costs are 'sticky' while revenue is not: operating leverage.")

# Ex2 - best diversifier
wide = uni.pivot(index="date", columns="ticker", values="close")
corr = wide.pct_change().dropna().corr()
avg_corr = corr.mean().sort_values()
print("\nBest diversifier:", avg_corr.index[0], round(avg_corr.iloc[0],3))
print(uni[uni.ticker==avg_corr.index[0]]["sector"].iloc[0], "- defensive sector, least tied to the market factor")

# Ex3 - tornado with cost inflation
def quick_value(revenue_growth=0.12, margin=0.085, multiple=18, cost_inflation=0.0):
    rev_next = fin["revenue_cr"].iloc[-1] * (1 + revenue_growth)
    eff_margin = margin - cost_inflation
    return rev_next * eff_margin * multiple
base = quick_value()
print(f"\nBase Rs {base:,.0f} cr; cost_inflation of just 1pp of margin -> Rs {quick_value(cost_inflation=0.01):,.0f} cr")
print("A 1pp margin hit rivals a 20% multiple swing - margin assumptions punch far above their size.")

## Streamlit exercises (from the module page)
These are modifications to `m4_price_dashboard.py` - run locally with `streamlit run`:

1. **Add a KPI**: up-day percentage -> `up = (d['ret']>0).mean()` then `col5.metric('Up days', f'{up*100:.0f}%')` (add a fifth column).
2. **Add a control**: `log_scale = st.sidebar.checkbox('Log scale')` then `if log_scale: ax.set_yscale('log')`.
3. **New tab**: wrap the two detail charts in `tab1, tab2 = st.tabs(['Distribution','Drawdown'])` and put one chart in each `with` block.